In [1]:
import pandas as pd
import os
import sys
import json
from pathlib import Path
import numpy as np

# Notebook is in notebooks/, so repo root is parent
REPO_ROOT = Path.cwd().parent
SRC_PATH = REPO_ROOT / "src"

# Insert src at the front of sys.path so imports work
sys.path.insert(0, str(SRC_PATH))

In [2]:
import importlib
import preprocessing.vitals_preprocessing as vp

importlib.reload(vp)

<module 'preprocessing.vitals_preprocessing' from '/Users/brandonng/Documents/GitHub/ClinicalDigitalTwin/src/preprocessing/vitals_preprocessing.py'>

In [ ]:
# Get repo root relative to the current notebook
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Load static preprocessing config
config_path = os.path.join(repo_root, "configs", "vitals_preprocessing_params.json")
with open(config_path, "r") as f:
    config = json.load(f)

# Set input and output directories
in_dir = os.path.join(repo_root, config["paths"]["in_dir"])

In [ ]:
lab_events = pd.read_csv('lab_events_vitals.csv')
lab_events['value'] = lab_events['value'].replace(['___', ''], pd.NA)
lab_events['valuenum'] = lab_events['valuenum'].fillna(
    pd.to_numeric(
        lab_events['value'].str.extract(r'([-+]?\d*\.?\d+)')[0],
        errors='coerce'
    )
)
lab_events_vitals = (
    lab_events
    .pivot_table(
        index=['subject_id', 'hadm_id', 'charttime'],
        columns='label',
        values='valuenum',
        aggfunc='first'
    )
    .reset_index()
)
lab_events_vitals = lab_events_vitals.rename(columns={
    'Oxygen Saturation': 'Lab Oxygen Saturation %',
    'Temperature': 'Lab Temperature (C)'
})
lab_events_vitals 

label,subject_id,hadm_id,charttime,Lab Oxygen Saturation %,Lab Temperature (C)
0,10001843,26133978.0,2134-12-05 19:30:00,NaN,36.6
1,10001843,26133978.0,2134-12-05 21:29:00,81.0,NaN
2,10001884,26184834.0,2131-01-11 06:37:00,81.0,36.1
3,10001884,26184834.0,2131-01-11 11:33:00,72.0,NaN
4,10001884,26184834.0,2131-01-11 22:21:00,NaN,36.9
...,...,...,...,...,...
230065,19999840,21033226.0,2164-09-15 02:46:00,95.0,NaN
230066,19999840,21033226.0,2164-09-15 16:30:00,95.0,NaN
230067,19999840,21033226.0,2164-09-17 08:11:00,93.0,NaN
230068,19999840,21033226.0,2164-09-17 13:34:00,25.0,NaN


In [ ]:
from preprocessing.vitals_preprocessing import preprocess_lab_events

labevents_vitals = preprocess_lab_events(in_dir)

In [21]:
ed_vitals = pd.read_csv('ed_vitals.csv')
ed_vitals['charttime'] = pd.to_datetime(ed_vitals['charttime'])
ed_vitals = ed_vitals.rename(columns={
    'temperature': 'ED Temperature (F)',
    'heartrate': 'ED Heart Rate',
    'resprate': 'ED Respitory Rate',
    'o2sat': 'ED Oxygen Saturation %',
    'sbp': 'ED sbp',
    'dbp': 'ED dbp'
})
ed_vitals

,subject_id,stay_id,charttime,ED Temperature (F),ED Heart Rate,ED Respitory Rate,ED Oxygen Saturation %,ED sbp,ED dbp
0,17195991,38649090,2110-01-11 21:45:00,NaN,141.0,24.0,94.0,NaN,NaN
1,17195991,38649090,2110-01-11 21:50:00,NaN,123.0,24.0,91.0,151.0,120.0
2,17195991,38649090,2110-01-11 22:00:00,NaN,133.0,23.0,99.0,180.0,86.0
3,17195991,38649090,2110-01-11 22:07:00,NaN,164.0,24.0,99.0,198.0,116.0
4,17195991,38649090,2110-01-11 22:23:00,NaN,130.0,16.0,100.0,235.0,126.0
...,...,...,...,...,...,...,...,...,...
1225234,11973788,33494247,2212-04-06 07:42:00,98.2,74.0,17.0,97.0,152.0,54.0
1225235,11973788,33494247,2212-04-06 09:01:00,98.2,84.0,16.0,99.0,132.0,89.0
1225236,11973788,33494247,2212-04-06 10:30:00,98.1,65.0,17.0,100.0,106.0,90.0
1225237,11973788,33494247,2212-04-06 10:44:00,98.1,67.0,16.0,98.0,140.0,42.0


In [ ]:
from preprocessing.vitals_preprocessing import preprocess_ed_vitals

ed_vitals = preprocess_ed_vitals(in_dir)

In [22]:
omr = pd.read_csv('omr_vitals.csv')
omr['result_name'] = omr['result_name'].replace('Weight', 'Hosp Weight (Lbs)')
omr['result_name'] = omr['result_name'].replace('Height', 'Hosp Height (Inches)')
omr['result_name'] = omr['result_name'].replace('BMI', 'Hosp BMI (kg/m2)')

omr_pivot = (
    omr
    .pivot_table(
        index=['subject_id', 'chartdate'],
        columns='result_name',
        values='result_value',
        aggfunc='first'   # important: avoids aggregation issues
    )
    .reset_index()
)
omr_pivot['chartdate'] = pd.to_datetime(omr_pivot['chartdate'])
omr_pivot = omr_pivot.rename(columns={
    'Blood Pressure': 'Hosp Blood Pressure'
})
omr_cleaned = (
    omr_pivot
    .groupby(['subject_id', 'chartdate'], as_index=False)
    .agg({
        'Hosp BMI (kg/m2)': 'first',
        'Hosp Height (Inches)': 'first',
        'Hosp Weight (Lbs)': 'first',
        'Hosp Blood Pressure': 'first'
    })
)
omr_cleaned

/tmp/ipykernel_3518/3617536169.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  omr = pd.read_csv('omr_vitals.csv')


result_name,subject_id,chartdate,Hosp BMI (kg/m2),Hosp Height (Inches),Hosp Weight (Lbs),Hosp Blood Pressure
0,10000032,2180-04-27,None,None,None,110/65
1,10000032,2180-05-07,None,None,None,None
2,10000032,2180-05-25,None,None,None,106/60
3,10000032,2180-06-01,None,None,None,121/77
4,10000032,2180-06-22,None,None,None,100/60
...,...,...,...,...,...,...
2229536,19999828,2148-02-26,None,None,None,115/79
2229537,19999828,2148-04-29,None,None,None,105/67
2229538,19999828,2148-07-22,None,None,None,104/76
2229539,19999828,2148-10-19,None,None,None,112/73


In [ ]:
from preprocessing.vitals_preprocessing import preprocess_omr

omr_data = preprocess_omr(in_dir)

In [23]:
# First, make sure time columns exist
omr_cleaned['chartdate'] = pd.to_datetime(omr_cleaned['chartdate'])
ed_vitals['charttime'] = pd.to_datetime(ed_vitals['charttime'])
ed_vitals['chartdate'] = ed_vitals['charttime'].dt.normalize()
lab_events_vitals['charttime'] = pd.to_datetime(lab_events_vitals['charttime'])
lab_events_vitals['chartdate'] = lab_events_vitals['charttime'].dt.normalize()

# Rename for clarity
ed_vitals = ed_vitals.rename(columns={'charttime': 'charttime_ed'})
lab_events_vitals = lab_events_vitals.rename(columns={'charttime': 'charttime_lab'})

ed_omr = ed_vitals.merge(
    omr_cleaned,
    on=['subject_id', 'chartdate'],
    how='right'
)

final = lab_events_vitals.merge(
    ed_omr,
    on=['subject_id', 'chartdate'],
    how='right'
)

# Now sorting works
final = final.sort_values(['subject_id', 'chartdate', 'charttime_lab', 'charttime_ed'])
final

,subject_id,hadm_id,charttime_lab,Lab Oxygen Saturation %,Lab Temperature (C),chartdate,stay_id,charttime_ed,ED Temperature (F),ED Heart Rate,ED Respitory Rate,ED Oxygen Saturation %,ED sbp,ED dbp,Hosp BMI (kg/m2),Hosp Height (Inches),Hosp Weight (Lbs),Hosp Blood Pressure
0,10000032,NaN,NaT,NaN,NaN,2180-04-27,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,110/65
1,10000032,NaN,NaT,NaN,NaN,2180-05-07,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None
2,10000032,NaN,NaT,NaN,NaN,2180-05-25,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,106/60
3,10000032,NaN,NaT,NaN,NaN,2180-06-01,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,121/77
4,10000032,NaN,NaT,NaN,NaN,2180-06-22,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,100/60
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2415173,19999828,NaN,NaT,NaN,NaN,2148-02-26,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,115/79
2415174,19999828,NaN,NaT,NaN,NaN,2148-04-29,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,105/67
2415175,19999828,NaN,NaT,NaN,NaN,2148-07-22,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,104/76
2415176,19999828,NaN,NaT,NaN,NaN,2148-10-19,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,112/73


In [ ]:
from preprocessing.vitals_preprocessing import merge_vitals

vitals_data = merge_vitals(omr_data, ed_vitals ,labevents_vitals)